<h1 id="toctitle">Regular expressions</h1>
<ul id="toc"/>

## What are regular expressions?

Regular expressions (aka. _regex_) are a special mini-language for describing patterns in strings. 

Handy for working with patterns in environmental sensor codes, station logs, and other text files.

Also crop up in other tools: text editors, grep.

### Regular expression module

The tools for using regular expressions live in the `re` module. We have to `import` the module, then use the module name when running functions:


In [6]:
import re
re.search('a', 'abc')

<re.Match object; span=(0, 1), match='a'>

In [8]:
search('a', 'abc')

NameError: name 'search' is not defined

## Searching for patterns with variation

`re.search()` takes two arguments: a pattern and a string. It searches for the pattern in the string and returns `True` or `False`. 

Here is a boring pattern:

In [10]:
record = "WWSSRCRSWRCWRS"

if re.search("RCR", record):
    print("pattern found!")

pattern found!



### Alternation

Here's an example of a pattern that's a bit more interesting. When there are two different possibilities we surround them with parentheses and separate with pipe characters:

In [12]:
record = "WWSSRCCRRCCSWRCWRS"

if re.search("CC(RR|SS)CC", record):
    print("pattern found!")

pattern found!


### Character groups

A very common type of alternation is when we want to allow any one of a list of characters. We can write it like this:

In [148]:
record = "WWSSRCCRRCCSWRCWRS"
if re.search("CC(RR|S|H)CC", record):
    print("pattern found!")


pattern found!


or with a shorthand like this:

In [150]:
record = "WWSSRCCRRCCSWRCWRS"
if re.search("CCC[RSH]CCC", record):
    print("pattern found!")


Sometimes it's easier to describe a character group by listing the characters that are __not__ allowed. Special rule: if the character group starts with ^ then it means any character except these ones:

In [28]:
record = "RCRXWRCWRS"

if re.search("[^RCSW]", record):
    print("unknown code found!")


unknown code found!


There are useful shortcuts for some commonly-used character groups:

- A full stop (aka dot, period) stands for any character
- `\d` means any digit
- `\s` means any whitespace (spaces, tabs)

### Quantifiers

Another type of variation is the number of times something is repeated.

A question mark means the thing preceding it is optional. In the pattern `CW?C` the W is optional. The pattern will match `CWC` and `CC`. 

A plus means that the thing preceding it can be repeated more than once. In the pattern `CW+C` the W can be repeated, so it matches `CWC`, `CWWC`, `CWWWC`, etc. but __not__ `CC`

An asterisk is the most flexible quantifier; the thing preceding it optional, but can also be repeated. The pattern `CW*C` will match `CC`, `CWC`, `CWWC`, `CWWWC`, etc. 

For more specificity, we can specify a minimum and maximum number of repetitions:

`CW{2,4}C` will match `CWWC`, `CWWWC` and `CWWWWC` but __not__ `CWC` or `CWWWWWC`, etc. 



### Positions

Unlike all the stuff above, __positions__ specify where the pattern has to match the string. 

`^` means the start (don't get confused: it also means a negated character group as described earlier). So `^H` will match `HCCR` but not `WCHR`. 

`$` means the end, so the pattern `H$` will match `CCRWH` but not `CHCR` 



### Combinations

The real power of regular expressions comes from combining all these features. Here's a complex regular expression that describes a full sensor record with a station code and a run of condition flags:

`^STN\d{3}:[RCSW]{10,50}$`

Look at the features:

- string must start with STN followed by three digits and a colon
- then between 10 and 50 flags that must be R/C/S/W
- string must end immediately after the flags

This would be possible to do without regular expressions, but much harder work!

## Other stuff we can do with regular expressions

### Extracting the match

In an `if` statement, `re.search()` behaves like a true/false function, but in fact it returns a re match object. We can get that match object and use methods to get information from it. For example let's find any non-clear day:


In [39]:
record = "CCCCCCRCCCCCCCCCCC"

if re.search("[^C]", record):
    print("non-clear day found!")


non-clear day found!


We know that we found an code that isn't `C`, but what was it? Calling `group()` on the match object will tell us:


In [43]:
record = "CCCCCCRCCCCCCCCCCC"
match = re.search("[^C]", record)

# m is now a match object
if match:
    print("non-clear day found!")
    print("the flag is " + match.group())


non-clear day found!
the flag is R


### Getting the position

Another thing we can do is get the position of the match with `start()` (also `end()`):

In [61]:
record = "CCCCCCRCCCCCCCCCCC"
match = re.search("[^C]", record)

# m is now a match object
if match:
    print("non-clear day found!")
    print("the flag is " + match.group())
    print("the day is " + str(match.start()))


non-clear day found!
the flag is R
the day is 6


### Splitting a string with a regex

`re.split()` works just like regular `split()`, but takes a regular expression pattern as the separator. Here we split a condition record whenever we see an unknown flag. Note pattern reuse!


In [63]:
record = "CWCCWRRRWWWCWCWRCCCWWRRWCWCWWWCCCWCW"

# find all intervals without rain
re.split("R+", record)

['CWCCW', 'WWWCWCW', 'CCCWW', 'WCWCWWWCCCWCW']

The output is a list of strings. 

Above, we exclude the bits of the string that matched the pattern and just keep the non-matching bits. For the opposite, use `re.findall()`. E.g. find all runs of rain and snow that are at least three days long:


In [71]:
record = "CWSWSCWCSWSRRSRRSWWCCSCWCCCCCCSWCSRRRSSRRSCCCSWSCC"
re.findall("[RS]{3,5}", record)

['SRRSR', 'SRRRS', 'SRRS']

### Finding multiple matches

Some problems require complete match objects for multiple matches - use `re.finditer()`. E.g. using the same pattern and sequence, what are the start/stop positions of all runs of R/S?

In [21]:
record = "CWSWSCWCSWSRRSRRSWWCCSCWCCCCCCSWCSRRRSSRRSCCCSWSCC"
re.finditer("[RS]{3,}", record)

for match in re.finditer("[RS]{3,}", record):
    run_start = match.start()
    run_end = match.end()
    day_codes = match.group()
    print(run_start, run_end, day_codes)


10 17 SRRSRRS
33 42 SRRRSSRRS


In [130]:
import re
station_ids = ["xkn59438", "yhdck2", "eihd39d9", "chdsye847", "hedle3455", "xjhd53e", "45da", "de37dp"]


for sid in station_ids:
    match = re.search("5", sid)
    if match:
        print(sid)

xkn59438
hedle3455
xjhd53e
45da


## Exercises

### Station IDs

Here's a list of made-up station IDs:

`xkn59438, yhdck2, eihd39d9, chdsye847, hedle3455, xjhd53e, 45da, de37dp`

Write a program that will print only the station IDs that satisfy the following criteria – treat each criterion separately:

- contain the number 5
- contain the letter d or e
- contain the letters d and e in that order
- contain the letters d and e in that order with a single letter between them
- contain both the letters d and e in any order
- start with x or y
- start with x or y and end with e
- contain three or more digits in a row
- end with d followed by either a, r or p

__Warning: this is another exercise where it's easy to get the wrong answer, so check your outputs.__


In [152]:
for sid in station_ids:
    match = re.search("[de]", sid)
    if match:
        print(sid)

yhdck2
eihd39d9
chdsye847
hedle3455
xjhd53e
45da
de37dp


In [157]:
for sid in station_ids:
    match = re.search("de", sid)
    if match:
        print(sid)

de37dp


In [160]:
for sid in station_ids:
    match = re.search("d.e", sid)
    if match:
        print(sid)

hedle3455


In [169]:
for sid in station_ids:
    match = re.search("(d*e)", sid)
    if match:
        print(sid)

eihd39d9
chdsye847
hedle3455
xjhd53e
de37dp


In [209]:
for sid in station_ids:
    match = re.search("\d{3}", sid)
    if match:
        print(sid)

xkn59438
chdsye847
hedle3455


In [213]:
for sid in station_ids:
    match = re.search("[arp]+d$", sid)
    if match:
        print(sid)
 

### Weather split

Look at the file *long_record.txt* which contains a made up weather record. 

We are interested in weather patterns which consist of two snowy days, followed by one day of any type of weather, followed by another two snowy days. 

How many times do we see that weather pattern occur in the record?

What is the average interval between these ocurrences i.e. how many days do we expect to wait after one such pattern until we see the next one? (hint: use the `sum` function to add numbers).

In [224]:
record = open('long_record.txt').read().rstrip('\n')

count = len(re.findall('SS.SS', record))

count

4

In [ ]:
interval_length = []

for interval in re.split('SS.SS', record)

